In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util
import torch, json
from tqdm import tqdm

# ---- Paths ----
A_PATH = "/home/ubuntu/TW_MultiLabel_SMP/jupyter-notebooks/training_sets/train_20251007_153631.csv"  # queries
B_PATH = "/home/ubuntu/TW_MultiLabel_SMP/datasets/1000_Posts_Annotations - Combined_Dataset.csv"           # reference pool
A_EMB_PATH = "/home/ubuntu/embeddings/r6_embeddings_A.pt"
B_EMB_PATH = "/home/ubuntu/embeddings/r6_embeddings_B.pt"
SIM_JSON_OUT = "/home/ubuntu/TW_MultiLabel_SMP/similarity_scores/cross_similar_posts_k3_500_r6.json"

# ---- Read ----
A = pd.read_csv(A_PATH)
B = pd.read_csv(B_PATH)

# ---- Build full_text (same style as your notebook) ----
A['full_text'] = A['title'].fillna('') + '. ' + A['selftext'].fillna('')
B['full_text'] = B.get('title','').fillna('') + '. ' + B.get('body','').fillna('')  # robust to missing cols

A.head(), B.head()


(        id        subreddit  \
 0  1ku7sfe    relationships   
 1  1cdk975    relationships   
 2  1l5ybv0         abortion   
 3   gsyxre         abortion   
 4  1lk8sba  TwoXChromosomes   
 
                                                title  \
 0  I don't know how to get over mourning my pregn...   
 1  I (26M) am not sure how to approach my wife (2...   
 2              Not bleeding after taking misoprostol   
 3  What is a legitimate reason to NEED to abort a...   
 4  The Answer to The Question, "When Was Your Las...   
 
                                             selftext          created_utc  \
 0  I'm not sure where to post this so I thought h...   2025-05-24 9:40:46   
 1  So I met my wife at twenty-one years old and s...  2024-04-26 12:34:48   
 2  Hi everyone, just a little background I took M...  2025-06-07 23:33:12   
 3  This may be against the rules but I'm getting ...  2020-05-29 18:58:34   
 4  "Fuck the fascists. I'm not telling." \n\nI ta...  2025-06-25 15:31:

In [2]:
model = SentenceTransformer("all-mpnet-base-v2")

# Encode and save A
r6_emb_A = model.encode(
    A['full_text'].tolist(),
    convert_to_tensor=True,
    show_progress_bar=True,
    normalize_embeddings=True
)
torch.save(r6_emb_A, A_EMB_PATH)

# Encode and save B (do once; later you can just torch.load(B_EMB_PATH))
r6_emb_B = model.encode(
    B['full_text'].tolist(),
    convert_to_tensor=True,
    show_progress_bar=True,
    normalize_embeddings=True
)
torch.save(r6_emb_B, B_EMB_PATH)


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

In [8]:
emb_A = torch.load(A_EMB_PATH)
emb_B = torch.load(B_EMB_PATH)

In [3]:
k_max = 3

# Cosine similarity matrix: [len(A), len(B)]
# (normalize_embeddings=True above ⇒ cosine == dot)
sim_mat = util.cos_sim(r6_emb_A, r6_emb_B)  # torch tensor

# Top-k along B axis for each A row
top_vals, top_idx = torch.topk(sim_mat, k=k_max, dim=1)  # shapes: [len(A), k]

# Pack to dict: {a_row_index: [(b_index, score), ...]}
similar_posts = {}
for i in range(top_idx.size(0)):
    indices = top_idx[i].tolist()
    scores  = top_vals[i].tolist()
    similar_posts[i] = list(zip(indices, scores))

# Save JSON
with open(SIM_JSON_OUT, "w") as f:
    json.dump(similar_posts, f)

print(f"Saved cross-sim results to {SIM_JSON_OUT}")

Saved cross-sim results to /home/ubuntu/TW_MultiLabel_SMP/similarity_scores/cross_similar_posts_k3_500_r6.json
